# Docker 入门与实践：够用，但不劝退

这份教程分为三层：

- **必学**：Docker 解决什么问题、镜像、容器、Dockerfile、`build`、`run`。
- **实践**：把 Python 程序放进容器，再运行一个可以从浏览器访问的 Web 服务。
- **选学**：环境变量、数据保存、Compose 和排错。

建议分两次学习：

1. 第一次学到“命令行 Python 程序”，约 25 分钟。
2. 第二次完成 Web 服务和选学内容，约 30 分钟。

不需要一次全部看完。

## 第一部分：先把核心逻辑弄懂（必学）

### 1. Docker 解决什么问题？

一个 Python 项目能运行，通常依赖：

- Python 版本；
- 第三方库及版本；
- 操作系统环境；
- 环境变量和启动命令。

你把代码交给同学，他的环境不同，程序可能无法运行。

Docker 的作用可以概括为：

> **把应用和运行应用所需的环境一起打包，再以统一方式启动。**

专业名称是：**容器化（Containerization）**。

### 2. 四个核心概念

```text
Dockerfile --docker build--> 镜像 --docker run--> 容器
   构建说明                  静态模板             运行实例
                                 ↑
                              镜像仓库
```

| 概念 | 英文 | 作用 |
|---|---|---|
| Dockerfile | Dockerfile | 描述如何制作镜像 |
| 镜像 | Image | 包含程序和环境的静态模板 |
| 容器 | Container | 镜像启动后的运行实例 |
| 镜像仓库 | Registry | 保存和分发镜像，例如 Docker Hub |

最容易被问到的区别：

> 镜像是模板，本身不运行；容器是镜像启动后的实例。同一个镜像可以启动多个容器。

In [ ]:
flow = {
    "Dockerfile": "构建说明",
    "Image": "静态模板",
    "Container": "运行实例",
    "Registry": "镜像仓库",
}

for name, meaning in flow.items():
    print(f"{name:12} -> {meaning}")

### 3. Docker 和虚拟机不是一回事

| 对比项 | Docker 容器 | 虚拟机 |
|---|---|---|
| 隔离对象 | 主要是应用进程 | 一整套虚拟计算机 |
| 操作系统 | 通常共享宿主机内核 | 通常拥有完整客体操作系统 |
| 启动速度 | 通常较快 | 通常较慢 |
| 体积 | 通常较小 | 通常较大 |

目前只需要知道：

> 容器不是一台完整的小电脑，它更像一个受到隔离和限制的程序运行空间。

底层的 namespaces、cgroups 暂时不展开。

### 4. 安装后先验证

在 PowerShell 中执行：

```powershell
docker --version
docker info
docker run --rm hello-world
```

- 第一条检查 Docker 命令。
- 第二条检查 Docker 引擎是否运行。
- 第三条下载镜像并启动容器，验证完整流程。

当前电脑尚未检测到 `docker` 命令。可以先学内容，实机练习需要安装并启动 Docker Desktop。

In [ ]:
import shutil

docker_path = shutil.which("docker")
if docker_path:
    print("Docker 已安装：", docker_path)
else:
    print("未检测到 Docker。安装前仍可阅读和运行本教程中的 Python 单元。")

### 5. 第一次运行容器

```powershell
docker run --rm python:3.12-slim python -c "print('Hello Docker')"
```

拆开看：

| 命令部分 | 含义 |
|---|---|
| `docker run` | 创建并启动一个新容器 |
| `--rm` | 程序结束后自动删除容器 |
| `python:3.12-slim` | 镜像名和标签 |
| `python -c ...` | 在容器内执行的命令 |

如果本地没有镜像，Docker 会先从镜像仓库下载。

## 第二部分：容器化自己的 Python 程序（必做实践）

### 6. 准备程序

创建目录 `hello-docker`，其中放两个文件：

```text
hello-docker/
├── hello.py
└── Dockerfile
```

`hello.py`：

```python
import os

name = os.getenv("USER_NAME", "同学")
print(f"你好，{name}！这个程序正在容器里运行。")
```

这里使用环境变量，让启动容器的人决定显示什么名字。

In [ ]:
def greeting(environment):
    name = environment.get("USER_NAME", "同学")
    return f"你好，{name}！这个程序正在容器里运行。"


print(greeting({}))
print(greeting({"USER_NAME": "小明"}))
assert "小明" in greeting({"USER_NAME": "小明"})

### 7. 编写 Dockerfile

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY hello.py .

CMD ["python", "hello.py"]
```

逐行解释：

| 指令 | 人话翻译 |
|---|---|
| `FROM` | 以已经装好 Python 的镜像为基础 |
| `WORKDIR` | 后续在容器的 `/app` 目录工作 |
| `COPY` | 把本地的 `hello.py` 复制进镜像 |
| `CMD` | 容器启动时默认运行这条命令 |

专业名称：**基础镜像（Base Image）**、**工作目录（Working Directory）**、**启动命令（Default Command）**。

In [ ]:
dockerfile = "\n".join([
    "FROM python:3.12-slim",
    "",
    "WORKDIR /app",
    "",
    "COPY hello.py .",
    "",
    'CMD ["python", "hello.py"]',
])

print(dockerfile)
assert "FROM python:3.12-slim" in dockerfile
assert "COPY hello.py ." in dockerfile

### 8. 构建镜像

在 `hello-docker` 目录中执行：

```powershell
docker build -t hello-docker:1.0 .
```

- `build`：构建镜像。
- `-t hello-docker:1.0`：镜像名是 `hello-docker`，标签是 `1.0`。
- 最后的 `.`：构建上下文是当前目录。

构建上下文可以简单理解为：Docker 构建时允许读取的文件范围。

查看镜像：

```powershell
docker images
```

### 9. 启动容器，并传入环境变量

```powershell
docker run --rm -e USER_NAME="小明" hello-docker:1.0
```

你应该看到：

```text
你好，小明！这个程序正在容器里运行。
```

其中：

- `-e` 用来传入环境变量；
- 修改名字不需要重新构建镜像；
- 修改 `hello.py` 后，需要重新执行 `docker build`。

专业名称：**配置外部化（Externalized Configuration）**。

In [ ]:
def make_run_command(image, environment=None, remove=True):
    parts = ["docker", "run"]
    if remove:
        parts.append("--rm")
    if environment:
        for key, value in environment.items():
            parts.extend(["-e", f"{key}={value}"])
    parts.append(image)
    return " ".join(parts)


command = make_run_command("hello-docker:1.0", {"USER_NAME": "xiaoming"})
print(command)
assert command == "docker run --rm -e USER_NAME=xiaoming hello-docker:1.0"

### 10. `build`、`run` 和 `start` 怎么区分？

| 命令 | 做什么 |
|---|---|
| `docker build` | 根据 Dockerfile 制作镜像 |
| `docker run` | 根据镜像创建一个新容器并启动 |
| `docker start` | 重新启动一个已经存在但停止了的容器 |

```text
代码或 Dockerfile 变了 -> build
想创建一个新容器       -> run
想重启旧容器           -> start
```

### 11. 容器的生命周期

```text
镜像 --run--> 运行中的容器 --stop--> 已停止容器 --rm--> 删除
```

常用命令：

```powershell
docker ps                 # 查看运行中的容器
docker ps -a              # 查看全部容器
docker stop 容器名         # 停止容器
docker start 容器名        # 再次启动旧容器
docker rm 容器名           # 删除已停止容器
docker logs 容器名         # 查看程序输出和报错
```

停止不等于删除，删除容器也不等于删除镜像。

## 第三部分：运行一个网页程序（实践进阶）

### 12. 为什么需要端口映射？

容器有自己的网络空间。容器里的程序监听 8000 端口，不代表浏览器可以直接访问它。

我们需要建立映射：

```text
你的电脑 8080 端口  ->  容器 8000 端口
```

命令格式：

```powershell
docker run -p 宿主机端口:容器端口 镜像名
```

例如：

```powershell
docker run -p 8080:8000 my-web:1.0
```

浏览器访问 `http://localhost:8080`。

口诀：**左边是你电脑的端口，右边是容器里程序的端口。**

In [ ]:
def explain_ports(mapping):
    host_port, container_port = mapping.split(":")
    return {
        "host_port": int(host_port),
        "container_port": int(container_port),
    }


ports = explain_ports("8080:8000")
print(ports)
assert ports == {"host_port": 8080, "container_port": 8000}

### 13. 最小 Python Web 服务

创建 `app.py`：

```python
import os
from http.server import BaseHTTPRequestHandler, HTTPServer

APP_NAME = os.getenv("APP_NAME", "Docker Web")

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        body = f"Hello from {APP_NAME}!\n".encode()
        self.send_response(200)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

HTTPServer(("0.0.0.0", 8000), Handler).serve_forever()
```

这里必须监听 `0.0.0.0`。如果只监听 `127.0.0.1`，通常只能在容器内部访问。

### 14. Web 服务的 Dockerfile

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY app.py .
EXPOSE 8000
CMD ["python", "app.py"]
```

构建和启动：

```powershell
docker build -t my-web:1.0 .
docker run --name my-web -d -p 8080:8000 -e APP_NAME="My App" my-web:1.0
```

- `-d`：在后台运行。
- `--name`：给容器起名字。
- `-p`：端口映射。
- `-e`：环境变量。

查看日志：

```powershell
docker logs my-web
```

## 第四部分：知道用途即可（选学）

### 15. 数据为什么需要单独保存？

容器适合被停止、删除和重新创建。只写在容器内部的数据，删除容器后可能一起消失。

需要长期保存的数据应放在：

- **绑定挂载（Bind Mount）**：映射宿主机的具体目录，开发时常用；
- **数据卷（Volume）**：由 Docker 管理存储位置，数据库数据常用。

示例只需要看懂：

```powershell
docker volume create app-data
docker run -v app-data:/data some-image
```

今天不要求实际操作。

### 16. Docker Compose 是做什么的？

当 `docker run` 参数很多，或者项目有多个服务，可以把启动配置写入 `compose.yaml`。

```yaml
services:
  web:
    build: .
    ports:
      - "8080:8000"
    environment:
      APP_NAME: "Compose Demo"
```

常用命令：

```powershell
docker compose up -d --build
docker compose logs -f
docker compose down
```

当前阶段只需知道：Compose 是**用配置文件管理容器启动方式**的工具。

In [ ]:
compose_config = {
    "service": "web",
    "build": ".",
    "ports": ["8080:8000"],
    "environment": {"APP_NAME": "Compose Demo"},
}

print(compose_config)
assert compose_config["ports"][0] == "8080:8000"

## 第五部分：出问题时怎么查？

### 17. 容器一启动就退出

先执行：

```powershell
docker ps -a
docker logs 容器名
```

容器中的主程序结束，容器也会停止。日志通常能看到 Python 报错。

### 18. 浏览器无法访问

按顺序检查：

1. `docker ps`：容器是否正在运行？
2. `docker logs 容器名`：程序是否报错？
3. `-p 8080:8000` 是否写对？
4. 程序是否监听 `0.0.0.0`？
5. 容器端口是否与程序端口一致？

### 19. 修改代码后没有变化

如果代码通过 `COPY` 放进镜像，修改代码后需要重新：

```powershell
docker build -t 镜像名 .
docker run ...
```

## 第六部分：练习

### 练习 1

用箭头写出 Dockerfile、镜像、容器之间的转换关系。

### 练习 2

解释下面命令的每一部分：

```powershell
docker run --name demo -d -p 9000:8000 -e MODE=dev my-app:1.0
```

### 练习 3

修改了 `app.py` 后，为什么不能只执行 `docker start`？

### 练习 4

容器正在运行，但网页打不开。请写出你会检查的前三项。

In [ ]:
# 请先自己填写，不需要一次写对。

docker_flow = ""
port_meaning = ""
first_debug_command = ""

print("构建流程：", docker_flow)
print("端口含义：", port_meaning)
print("第一条排错命令：", first_debug_command)

## 练习参考答案

1. `Dockerfile --docker build--> 镜像 --docker run--> 容器`。
2. 创建名为 `demo` 的后台容器，把宿主机 9000 映射到容器 8000，传入 `MODE=dev`，使用 `my-app:1.0` 镜像。
3. `docker start` 只启动旧容器，旧容器仍来自旧镜像；代码修改后应重新构建镜像并创建新容器。
4. 先看 `docker ps`，再看 `docker logs 容器名`，然后核对端口映射和监听地址。

## 面试表达

> Docker 是一种容器化工具。我们使用 Dockerfile 描述应用环境并构建镜像，再通过镜像创建容器。镜像是静态模板，容器是运行实例。项目中可以通过端口映射对外提供服务，通过环境变量注入配置，通过数据卷持久化数据，并用 Docker Compose 管理启动配置。

这段话不需要现在背。完成一次 `build` 和 `run` 后，再回来读会自然很多。

## 最终复盘：掌握到什么程度算合格？

### 第一阶段合格线

- 能解释镜像和容器的区别。
- 能看懂一个简单 Dockerfile。
- 能独立执行 `docker build` 和 `docker run`。

### 第二阶段合格线

- 能解释 `-p 8080:8000`。
- 会使用环境变量。
- 出问题会先看 `docker ps -a` 和 `docker logs`。

### 暂时不要求

- namespaces、cgroups；
- 多阶段构建；
- Kubernetes；
- 复杂 Compose 编排。

先真正运行一个自己的 Python 容器，比背十页概念更有用。